# Kiểm tra VS Code kết nối Google Colab

Mở notebook trong VS Code. Chọn **Select Kernel > Colab > New Colab Server**, đăng nhập đúng tài khoản Google AI Pro và chọn máy có GPU. Kiểm tra kernel đang thuộc Colab trước khi chạy từng cell bằng nút tam giác bên trái.

Notebook chỉ đọc thông tin môi trường/GPU và kiểm tra Drive. Không cài thư viện, tải dataset hoặc huấn luyện model. File notebook ở Windows; các cell phải thực thi trên máy chủ Colab. Không yêu cầu Python của Colab trùng Python 3.13 của ứng dụng local.

Hướng dẫn chính thức: https://github.com/googlecolab/colab-vscode/wiki/User-Guide

## 1. Kiểm tra môi trường

Kết quả mong đợi: Linux, có thư mục /content và thông tin Python của máy chủ. Nếu cell báo lỗi, kiểm tra lại **Select Kernel**; chưa chuyển sang bước GPU.

In [2]:
import os
import platform
import shutil
import sys
from pathlib import Path

colab_environment_checked = False
if platform.system() != 'Linux' or not Path('/content').is_dir():
    raise RuntimeError('Môi trường chưa phù hợp Colab. Chọn Select Kernel > Colab, không chọn Python local.')
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError('Thiếu môi trường google.colab. Kiểm tra lại kernel; không cài gói này vào Windows để vượt kiểm tra.') from exc

colab_environment_checked = True
disk = shutil.disk_usage('/content')
print('Hệ điều hành:', platform.system())
print('Python:', sys.version.split()[0])
print('Python executable:', sys.executable)
print('Thư mục hiện tại:', os.getcwd())
print('CPU logic được hệ điều hành báo cáo:', os.cpu_count())
print(f'Ổ tạm Colab: tổng {disk.total / 2**30:.1f} GiB, còn trống {disk.free / 2**30:.1f} GiB')
print('Môi trường phù hợp Colab. Đối chiếu thêm tên kernel Colab trong VS Code.')

Hệ điều hành: Linux
Python: 3.13.16
Python executable: /usr/bin/python3
Thư mục hiện tại: /content
CPU logic được hệ điều hành báo cáo: 2
Ổ tạm Colab: tổng 112.6 GiB, còn trống 112.5 GiB
Môi trường phù hợp Colab. Đối chiếu thêm tên kernel Colab trong VS Code.


## 2. Kiểm tra GPU

Cell cần PyTorch có sẵn trên runtime và CUDA hoạt động. Nếu thiếu GPU, chọn lại cấu hình máy Colab. Không tự cài CUDA/PyTorch trên Windows và không chuyển sang train bằng CPU local.

In [1]:
!nvidia-smi

Tue Nov  3 14:34:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
if not globals().get('colab_environment_checked', False):
    raise RuntimeError('Chạy thành công cell kiểm tra môi trường trước.')

try:
    import torch
except ImportError as exc:
    raise RuntimeError('Runtime Colab này chưa có PyTorch. Ghi lại lỗi để xử lý dependency trên cloud.') from exc

print('PyTorch:', torch.__version__)
print('CUDA khả dụng:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Colab chưa cấp GPU hoạt động cho PyTorch. Kiểm tra cấu hình server và tài nguyên tài khoản.')

device = torch.cuda.get_device_properties(0)
print('GPU:', device.name)
print(f'VRAM tổng: {device.total_memory / 2**30:.1f} GiB')
print('GPU_OK — đây là kiểm tra tài nguyên, chưa có training hoặc benchmark.')

PyTorch: 2.11.0+cu130
CUDA khả dụng: True
GPU: Tesla T4
VRAM tổng: 14.6 GiB
GPU_OK — đây là kiểm tra tài nguyên, chưa có training hoặc benchmark.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Kết nối Drive bằng extension

Nhấn **Ctrl+Shift+P**, chọn **Colab: Mount Google Drive to Server...**. Extension thêm một cell mới ở cuối notebook. **Chạy cell được thêm đó**, hoàn tất đăng nhập/cấp quyền, rồi quay lại chạy cell kiểm tra Drive bên dưới. Không cần cài Google Drive for desktop hoặc đồng bộ dataset xuống Windows.

Nếu thư mục Drive chưa xuất hiện, kiểm tra kết quả cell mount. Dung lượng Drive và ổ tạm /content, RAM, VRAM là các tài nguyên riêng.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [6]:
if not globals().get('colab_environment_checked', False):
    raise RuntimeError('Chạy thành công cell kiểm tra môi trường trước.')

drive_root = Path('/content/drive/MyDrive')
if not os.path.ismount('/content/drive') or not drive_root.is_dir():
    raise RuntimeError('Chưa thấy Drive tại /content/drive/MyDrive. Chạy cell mount do extension thêm, rồi quay lại cell này.')

print('DRIVE_OK:', drive_root)
print('Đường dẫn dự kiến cho kết quả:', drive_root / 'BottleQC')
print('Chưa tạo thư mục, tải dữ liệu hoặc chạy training.')

DRIVE_OK: /content/drive/MyDrive
Đường dẫn dự kiến cho kết quả: /content/drive/MyDrive/BottleQC
Chưa tạo thư mục, tải dữ liệu hoặc chạy training.


## Tiếp theo

Sau khi có GPU_OK và DRIVE_OK, giữ thông tin GPU, VRAM và Python để chuẩn bị workflow detection. Code .py có thể soạn trên VS Code và đưa lên bằng Upload to Colab; cần cập nhật lại file trên server mỗi khi sửa bản local. Notebook training cũ chưa hoàn thiện và chưa được kiểm chứng end to end.

Khi xong phiên làm việc, dùng **Colab: Remove Server** để giải phóng máy chủ không còn cần dùng. Kernel Colab chạy code trên cloud; terminal PowerShell bình thường vẫn chạy trên Windows.